# RME Process Chatbot — live end-to-end demo

The whole backend, run start to finish on this machine, ending in a prompt you can type into —
in English or in Arabic.

```
 ../processes_pdf/*.pdf
        |  §1  extract      PyMuPDF native text, OCR fallback  ->  OCR-health audit only
        |
        |  §2  chunk        layout-based boundaries + MiniLM section labels
        v                   (reads the PDFs directly - layout isn't in extracted text)
 114 chunks + metadata + audit
        |  §3  index        BM25 (sparse) + all-MiniLM-L6-v2 -> FAISS (dense)
        v
 hybrid retriever
        |  §4  whitelist    every form number that really occurs in the corpus
        v
 §5  ask(question)
        |
        |     Arabic question?  -> translate AR->EN        (§5.5)
        v
      route -> retrieve top-3 -> qwen3:14b -> strip reasoning -> validate -> cite
        |
        |     Arabic question?  -> translate EN->AR
        v
      answer + sources + latency breakdown
```

**The model is `qwen3:14b`** and it is the only model this notebook runs. It was chosen on the
36-question eval whose results are in `model_eval_results.json`: 100% overall, 100% on the
`unanswerable` category, zero flagged citations. §5.5 explains what that choice does and does
not settle.

Arabic questions take the same path as English ones — retrieval, routing and the form-number
validator all operate on English throughout, and only the two ends translate. Details in §5.5.

Chunking is `adaptive_chunker.py`, which detects section boundaries from page layout rather than
from a list of expected heading strings.

§1 writes to `demo_extracted_raw.json` rather than the shipped `extracted_raw.json`, so no
artifact is overwritten; §2 re-chunks the PDFs and checks the result against the shipped
`chunks.json`.

**Run All**, then go to **§5.2** and put your own question in.

## 0. Setup

In [1]:
import json, sys, time, re, platform, textwrap
from pathlib import Path
from collections import Counter

# A Windows console is cp1252 and cannot encode Arabic at all. Jupyter is UTF-8
# already; this only matters when the notebook is executed headlessly through
# such a console (nbconvert from cmd.exe, say).
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

def _find_files_dir():
    """Locate the 'files' folder by marker file, not by assuming the CWD."""
    here = Path.cwd()
    for c in [here, here / "files", *here.parents]:
        if (c / "eval_set.json").exists() and (c / "retriever.py").exists():
            return c.resolve()
    raise SystemExit(f"Could not locate the 'files' folder from {here}")

FILES = _find_files_dir()
PDF_DIR = FILES.parent / "processes_pdf"
sys.path.insert(0, str(FILES))

import requests

OLLAMA_HOST = "http://localhost:11434"
MODEL = "qwen3:14b"     # the chosen model; see model_eval_results.json

print("files/       :", FILES)
print("processes_pdf:", PDF_DIR, "|", len(list(PDF_DIR.glob("*.pdf"))), "PDFs")
print("python       :", platform.python_version(), "|", platform.platform())

def ollama_models():
    r = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
    r.raise_for_status()
    return [m["name"] for m in r.json().get("models", [])]

try:
    have = ollama_models()
    ver = requests.get(f"{OLLAMA_HOST}/api/version", timeout=5).json()["version"]
    print(f"ollama {ver}  : up")
    print(f"  {MODEL:<12} {'pulled' if MODEL in have else 'MISSING -> ollama pull ' + MODEL}")
except Exception as e:
    print(f"ollama       : DOWN ({type(e).__name__}). Start `ollama serve` and re-run "
          "this cell. Sections 1-4 work without it; section 5 does not.")

files/       : C:\Users\dahab\OneDrive\Documents\GitHub\chatbot_handoff\files
processes_pdf: C:\Users\dahab\OneDrive\Documents\GitHub\chatbot_handoff\processes_pdf | 9 PDFs
python       : 3.14.4 | Windows-11-10.0.26200-SP0


ollama 0.32.5  : up
  qwen3:14b    pulled


## 1. Extract — PyMuPDF primary, OCR fallback

Real PDF bytes off disk. OCR fires only when a page's native text looks broken (under 20
characters, or an alphabetic ratio below 0.4) — what a genuinely scanned page looks like.

**This stage is no longer upstream of chunking.** §2 reads the PDFs directly, because the font
and layout signals it detects on exist only in the PDF and not in extracted text. What §1 is
now is the **OCR-health audit**, and that job matters more rather than less: §2 has no OCR path,
so any page without embedded text is a page the chunker cannot see. This cell is what tells you
which pages those are.

**The 9 `ocr_failed` lines below are expected, not a broken run.** Page 1 of every document is
an image-only signature/approval cover page. With no `tesseract` binary installed the fallback
raises, the page is logged loudly, and extraction continues on native text. No process content
lives on those cover pages — they carry the title, doc code and approval signatures, all of
which also appear in the document body. §2's `pages with no text` counter reports the same 9
pages from the other side.

In [2]:
import extract_pipeline as EX

DEMO_RAW = FILES / "demo_extracted_raw.json"
DEMO_LOG = FILES / "demo_ocr_fallback_log.json"

t0 = time.perf_counter()
extracted = EX.run(PDF_DIR, DEMO_RAW, DEMO_LOG, allow_ocr=True)
print(f"\nextraction wall time: {time.perf_counter() - t0:.1f}s")

PCM01_Customer_Satisfaction_Process_1.pdf: 6 pages, 5 native, 1 FELL BACK


PCM02_Branding_for_Construction_Sites_Process_1.pdf: 4 pages, 3 native, 1 FELL BACK


PCN01_Subcontract_Agreement_Process_1.pdf: 11 pages, 10 native, 1 FELL BACK


PQD12_Quality_Plan_Inspection_and_Testing_Process.pdf: 8 pages, 7 native, 1 FELL BACK


PSE01_RME_SelfExecution_Process_1.pdf: 5 pages, 4 native, 1 FELL BACK


PTN01_Project_Initiation_Process.pdf: 6 pages, 5 native, 1 FELL BACK


PTN02_Project_Launching_Process.pdf: 7 pages, 6 native, 1 FELL BACK


PVMO01_Vendor_selection_and_Bidding_Process.pdf: 8 pages, 7 native, 1 FELL BACK


PVMO02_Procurement_Process.pdf: 8 pages, 7 native, 1 FELL BACK

9 docs, 63 pages in 8.4s
OCR fallback pages: 9/63 (14.3%)
Saved  C:\Users\dahab\OneDrive\Documents\GitHub\chatbot_handoff\files\demo_extracted_raw.json
Log    C:\Users\dahab\OneDrive\Documents\GitHub\chatbot_handoff\files\demo_ocr_fallback_log.json

extraction wall time: 8.4s


## 2. Chunk — structural boundaries, semantic labels

Fixed-size windows would cut a process step away from its own form number, which is exactly
what most questions ask for. So chunking follows the document's own sections. `adaptive_chunker.py`
does that in two independent stages:

- **Where a section starts** is decided from *layout* — PyMuPDF gives font size and bold flags
  per span, and a heading is a short line that is visually distinct from body text. Where layout
  is flat (several of these documents style headings identically to body text) it falls back to
  the numbering pattern `1. SOMETHING`. Neither signal reads the heading's wording.
- **What the section is** is decided by comparing the heading to a canonical taxonomy: exact
  match, then fuzzy match, then MiniLM cosine — reusing the embedding model the retriever
  already loads. That is what absorbs `STAKHOLDER`, `PROCES INPUT` and `Process Operations`
  without a keyword list naming each one.

This replaced an earlier chunker that matched literal strings from a hand-written
`SECTION_HEADERS` list and therefore did both jobs at once. It worked on these 9 documents and
failed silently on anything else — a document saying `RESPONSIBLE PARTIES` instead of
`STAKEHOLDER` matched nothing, produced one document-sized chunk, and reported success.

Every chunk still carries `doc_code`, `title`, `filename`, `section` and `step`, plus the raw
heading and which tier labeled it, so an answer can cite where it came from and the chunking
itself is auditable.

**Two things this stage does not do.** It has no OCR path — it reads PDF layout, which only
exists for native text, so the 9 image-only cover pages from §1 are simply not chunked (the
`pages with no text` counter below is that fact, stated out loud). And `step` is `None` on every
chunk: step-level splitting inside `PROCESS OPERATION` is not implemented. Neither matters at 9
documents; the second is the one that bites at 500–600, where a section grows long enough that a
step and its form number land in different chunks.

In [3]:
import adaptive_chunker as AC
from retriever import route_prefixes          # doc-code routing lives with retrieval

t0 = time.perf_counter()
chunks, chunk_report = AC.chunk_corpus(PDF_DIR)
print(f"{len(chunks)} chunks from {chunk_report['docs']} documents "
      f"in {time.perf_counter() - t0:.1f}s")

# Chunking the PDFs here should reproduce the shipped artifact exactly.
shipped = json.load(open(FILES / "chunks.json", encoding="utf-8"))
identical = [c["text"] for c in chunks] == [c["text"] for c in shipped]
print(f"identical to shipped chunks.json ({len(shipped)} chunks): {identical}")

detectors = Counter()
for r in chunk_report["per_doc"]:
    detectors.update(r["detection"]["detectors"])

print(f"\nboundary detectors : {dict(detectors)}")
print(f"label methods      : {chunk_report['label_methods']}")
print(f"windowed fallback  : {chunk_report['docs_windowed_fallback']}/{chunk_report['docs']} docs")
print(f"unmatched headings : {len(chunk_report['unmatched_headings'])}")
print(f"pages with no text : {chunk_report['pages_with_no_native_text']}"
      "  (the image-only cover pages from §1 - no OCR path, so not chunked)")

print("\nchunks per document:")
for fn, n in Counter(c["filename"] for c in chunks).most_common():
    print(f"  {n:>3}  {fn}")

print("\nsections found:", ", ".join(sorted({c["section"] for c in chunks})))

ex = next((c for c in chunks if "PROCESS OPERATION" in c["section"]), chunks[0])
print(f"\nexample chunk -- {ex['filename']}")
print(f"  section={ex['section']!r}  from heading {ex['raw_heading']!r}")
print(f"  labeled by {ex['label_method']} ({ex['label_score']})  doc_code={ex['doc_code']}")
print("  " + textwrap.shorten(ex["text"], 300))

C:\Users\dahab\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4860.25it/s]

114 chunks from 9 documents in 14.7s
identical to shipped chunks.json (114 chunks): True

boundary detectors : {'numbering': 103, 'lone_number+stitch': 8, 'font_size': 3}
label methods      : {'front_matter': 9, 'exact': 80, 'fuzzy': 25}
windowed fallback  : 0/9 docs
unmatched headings : 0
pages with no text : 9  (the image-only cover pages from §1 - no OCR path, so not chunked)

chunks per document:
   13  PCM01_Customer_Satisfaction_Process_1.pdf
   13  PCN01_Subcontract_Agreement_Process_1.pdf
   13  PQD12_Quality_Plan_Inspection_and_Testing_Process.pdf
   13  PSE01_RME_SelfExecution_Process_1.pdf
   13  PTN02_Project_Launching_Process.pdf
   13  PVMO01_Vendor_selection_and_Bidding_Process.pdf
   12  PCM02_Branding_for_Construction_Sites_Process_1.pdf
   12  PTN01_Project_Initiation_Process.pdf
   12  PVMO02_Procurement_Process.pdf

sections found: DOCUMENT CHANGE HISTORY, FRONT MATTER, LIABLE STAKEHOLDER, OBJECTIVES, PERFORMANCE MEASURES, PROCESS CONTROL, PROCESS INPUT, PROCESS OPE

## 3. Index — hybrid BM25 + dense, with doc-code routing

Two channels, min-max normalised and mixed at `dense_weight=0.4`:

- **BM25** is the exact-match channel. Form numbers are literal strings; a dense model has no
  reason to keep `F-P-CN-01-11` and `F-P-VMO-01-01` apart.
- **MiniLM → FAISS `IndexFlatIP`** is the paraphrase channel ("who signs off on" ≈ "approved by").

Before either runs, `route_prefixes()` narrows the candidate pool to a single process family
when the query unambiguously names one. On the eval set that fires on 22 of 36 questions and
never once excluded the gold document.

Nothing downloads here: MiniLM loads from the local HF cache and chunk embeddings are cached in
`.embed_cache/` keyed by a corpus+model fingerprint. The slow part below is importing torch,
not encoding.

Note for §5.5: every one of these channels is English-only — BM25 tokenises `[a-z0-9]+`, MiniLM
was trained on English, and `PREFIX_HINTS` is an English keyword table. That is the reason
Arabic is handled by translating the query rather than by touching anything here.

In [4]:
from retriever import Retriever
import validator as V

t0 = time.perf_counter()
R = Retriever(chunks)      # weighted fusion, dense_weight=0.4, routing on
print(f"index built in {time.perf_counter() - t0:.1f}s")
print(f"model={R.model_name}  embeddings={R.embeddings.shape}  "
      f"fusion={R.fusion} (dense_weight={R.dense_weight})")

probe = "What form is used for the Customer Satisfaction Survey?"
print(f"\nroute for {probe!r} -> {route_prefixes(probe) or 'no route, search everything'}")
for h in R.search(probe, top_k=3):
    print(f"  [{h['score']:.3f}] {h['filename'][:44]:<44} {h['section'][:26]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11257.97it/s]

index built in 0.3s
model=all-MiniLM-L6-v2  embeddings=(114, 384)  fusion=weighted (dense_weight=0.4)

route for 'What form is used for the Customer Satisfaction Survey?' -> ['PCM']


  [0.937] PCM01_Customer_Satisfaction_Process_1.pdf    PROCESS OPERATION
  [0.896] PCM01_Customer_Satisfaction_Process_1.pdf    PROCESS INPUT
  [0.845] PCM01_Customer_Satisfaction_Process_1.pdf    RELATED DOCUMENTED INFORMA


## 4. Validator — deterministic hallucination proxy

No model judges another model. Pull every form-shaped string out of an answer with a
deliberately *loose* regex (so a malformed invention like `F-P-CM-1-1` is caught too), then
test each against the set of form numbers that actually occur in the corpus.

It **flags, it does not block.** At 9 documents the whitelist is a subset of reality — the
corpus references form families (FW, HR, OP, PU, QP) whose source documents aren't here yet —
so "unknown" today means *unverifiable*, not *fabricated*. `policy="block"` becomes safe once
the full 500–600 document corpus is indexed.

In [5]:
FORM_WHITELIST = V.build_form_whitelist(chunks)
DOC_WHITELIST = V.build_doc_code_whitelist(chunks)
print(f"{len(FORM_WHITELIST)} known form numbers, {len(DOC_WHITELIST)} known doc codes")

for p in [
    "The Customer Satisfaction Survey uses form F-P-CM-01-01.",   # real
    "Use form F-P-CM-03-07 to request a company car.",            # invented
    "Not specified in these process documents.",                  # no citation
]:
    v = V.validate_answer(p, FORM_WHITELIST, DOC_WHITELIST)
    print(f"  {v['verdict']:<8} cited={v['cited']} unknown={v['unknown']}   <- {p[:52]}")

67 known form numbers, 9 known doc codes
  clean    cited=['F-P-CM-01-01'] unknown=[]   <- The Customer Satisfaction Survey uses form F-P-CM-01
  flagged  cited=['F-P-CM-03-07'] unknown=['F-P-CM-03-07']   <- Use form F-P-CM-03-07 to request a company car.
  clean    cited=[] unknown=[]   <- Not specified in these process documents.


## 5. The chatbot

`ask()` is the whole pipeline in one call: detect the language → translate to English if the
question is Arabic → route → retrieve top-3 → build a context-only prompt → generate → strip
any reasoning trace → validate the citations → translate the answer back if the question was
Arabic → print it with its sources and a latency breakdown.

Generation is deterministic (`temperature=0`, `seed=0`), so a repeated question gives a
repeated answer. The prompt forbids answering from parametric memory and requires a filename
citation — that instruction is what makes the refusal in §5.4 work.

**Answers run to completion — `num_predict: -1`.** This used to be capped at 200 tokens, which
was invisible on every question in this notebook and in the eval set, because a form number, a
deadline or a refusal all finish well under it. An explanatory question does not: *"explain the
process operations in procurement"* hit the cap and stopped mid-sentence, with nothing in the
output saying so. The cap is a *maximum*, not a target — short answers still stop on their own,
so removing it costs nothing on the questions above and only spends time when an answer
genuinely needs the room.

What bounds generation now is `num_ctx`, set explicitly to 8192. Leaving it unset would take
Ollama's default (4096) and quietly reintroduce a ceiling that shrinks as `top_k` or chunk size
grows. The remaining trade is honest: with no output cap, a vague question has no ceiling at
all, and on CPU that can run for minutes with only the 1800s HTTP timeout as a backstop.

`strip_reasoning()` earns its place: `qwen3:14b` is a reasoning model. Requests set
`think: False` to suppress traces at the source, and anything that still slips through is cut
before validation — otherwise a model *musing* "it might be F-P-CM-01-01" would register as a
confident citation it never actually made. The same wrapper is handed to `translate.py`, so a
trace can't leak into a translation either.

**Validation runs on the English answer, before translation.** `FORM_RE` in `validator.py` is
Latin-script, and the check has to see what the model actually emitted rather than a paraphrase
of it. §5.5 has the rest of the Arabic design.

In [6]:
from translate import is_arabic, to_english, to_arabic

THINK_RE = re.compile(r"<think>.*?</think>", re.S | re.I)

def strip_reasoning(text: str) -> str:
    """Remove <think>...</think> traces before validation."""
    out = THINK_RE.sub("", text or "")
    if "<think>" in out.lower():          # unterminated trace: drop the dangling tail
        out = re.split(r"<think>", out, flags=re.I)[0]
    return out.strip()

def build_prompt(question, retrieved):
    context = "\n\n".join(f"[{c['filename']} | {c['section']}]\n{c['text']}" for c in retrieved)
    return (
        "Answer using ONLY the context below. If the answer isn't in the context, "
        "say 'Not specified in these process documents.' Always cite the doc filename.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {question}\nANSWER:"
    )

def generate(prompt, model=MODEL, timeout=1800):
    payload = {
        "model": model, "prompt": prompt, "stream": False,
        "options": {"temperature": 0, "num_predict": -1, "num_ctx": 8192, "seed": 0},
        "think": False,
    }
    resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=timeout)
    if resp.status_code == 400:           # older Ollama, or the model rejects `think`
        payload.pop("think")
        resp = requests.post(f"{OLLAMA_HOST}/api/generate", json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()["response"].strip()

# translate.py never talks to Ollama itself - it takes a generator. Passing this
# one means translations get the same determinism and the same trace stripping.
def _gen(prompt):
    return strip_reasoning(generate(prompt))

def ask(question, top_k=3, show_context=False, quiet=False):
    """Full pipeline for one question, English or Arabic. Returns the record."""
    src_ar = is_arabic(question)

    t0 = time.perf_counter()
    q_en = to_english(question, _gen) if src_ar else question
    t_in = time.perf_counter() - t0

    # Everything from here to validation runs on English, exactly as before.
    t0 = time.perf_counter()
    route = route_prefixes(q_en)
    hits = R.search(q_en, top_k=top_k)
    t_ret = time.perf_counter() - t0

    t0 = time.perf_counter()
    raw = generate(build_prompt(q_en, hits))
    t_gen = time.perf_counter() - t0

    answer = strip_reasoning(raw)
    v = V.validate_answer(answer, FORM_WHITELIST, DOC_WHITELIST)

    # display_answer, not answer: under policy="block" this is the refusal text,
    # and an Arabic user should get the refusal in Arabic too.
    t0 = time.perf_counter()
    display = to_arabic(v["display_answer"], _gen) if src_ar else v["display_answer"]
    t_out = time.perf_counter() - t0

    rec = {"question": question, "question_en": q_en, "lang": "ar" if src_ar else "en",
           "model": MODEL, "answer": answer, "display": display, "raw": raw,
           "hits": hits, "validator": v, "had_reasoning": raw != answer,
           "translate_in_s": t_in, "retrieval_s": t_ret,
           "generation_s": t_gen, "translate_out_s": t_out}
    if quiet:
        return rec

    print("=" * 78)
    print(f"Q: {question}")
    print("=" * 78)
    print(textwrap.fill(display, 78, initial_indent="  ", subsequent_indent="  "))
    print(f"\n  sources (top-{top_k}, route={route or 'none'}):")
    for h in hits:
        step = f"  |  step {h['step']}" if h["step"] else ""
        print(f"    [{h['score']:.3f}] {h['filename']}  |  {h['section']}{step}")
    flag = "clean" if v["ok"] else f"FLAGGED - unverifiable form(s): {', '.join(v['unknown'])}"
    print(f"\n  cited forms : {v['cited'] or 'none'}   validator: {flag}")

    lat = f"retrieval {t_ret*1000:.0f} ms + generation {t_gen:.1f} s"
    if src_ar:
        lat = f"translate-in {t_in:.1f} s + {lat} + translate-out {t_out:.1f} s"
    print(f"  latency     : {lat} = {t_in + t_ret + t_gen + t_out:.1f} s   ({MODEL})")
    if rec["had_reasoning"]:
        print("  note        : a reasoning trace was stripped before validation")
    if show_context:
        if src_ar:
            print(f"\n  --- question as translated for retrieval ---\n    {q_en}")
            print(f"  --- answer as validated, before translation ---")
            print(textwrap.indent(textwrap.fill(answer, 74), "    "))
        print("\n  --- context sent to the model ---")
        for h in hits:
            print(f"\n  [{h['filename']} | {h['section']}]")
            print(textwrap.indent(textwrap.fill(h["text"], 74), "    "))
    print()
    return rec

### 5.1 Worked examples

Three shapes the corpus supports — a form lookup, a numeric deadline, and a role lookup. The
first call also pays the model's cold-load cost (`qwen3:14b` is ~9 GB off disk, and nothing is
on GPU here), so its latency is the outlier; re-run the cell and it drops.

Every answer prints with the chunks it was built from and the validator's verdict. That pairing
is the whole point of the layout: an answer on its own cannot tell you whether a wrong result
was a retrieval failure or a generation failure, and those have completely different fixes. A
refusal in particular looks like the system being careful — the only way to know is to read the
sources printed underneath it.

In [7]:
_ = ask("What form is used for the Customer Satisfaction Survey?")
_ = ask("Within how many hours must the PM send corrective action plans "
        "after a customer satisfaction gap is reported?")
_ = ask("Who is responsible for issuing an NCR when there is a gap between "
        "customer perception and RME's expected standard?")

Q: What form is used for the Customer Satisfaction Survey?
  The form used for the Customer Satisfaction Survey is **Customer
  Satisfaction Survey, F-P-CM-01-01**.   (Reference:
  PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [0.937] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.896] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS INPUT
    [0.845] PCM01_Customer_Satisfaction_Process_1.pdf  |  RELATED DOCUMENTED INFORMATION

  cited forms : ['F-P-CM-01-01']   validator: clean
  latency     : retrieval 21 ms + generation 69.0 s = 69.0 s   (qwen3:14b)



Q: Within how many hours must the PM send corrective action plans after a customer satisfaction gap is reported?
  The PM must send corrective action plans within 24 hours after a customer
  satisfaction gap is reported. (Doc filename:
  PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [1.000] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.675] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS CONTROL
    [0.572] PCM01_Customer_Satisfaction_Process_1.pdf  |  OBJECTIVES

  cited forms : none   validator: clean
  latency     : retrieval 436 ms + generation 15.9 s = 16.3 s   (qwen3:14b)



Q: Who is responsible for issuing an NCR when there is a gap between customer perception and RME's expected standard?
  QA is responsible for issuing an NCR when there is a gap between customer
  perception and RME's expected standard. (Doc filename:
  PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [0.923] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.760] PCM01_Customer_Satisfaction_Process_1.pdf  |  OBJECTIVES
    [0.626] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS CONTROL

  cited forms : none   validator: clean
  latency     : retrieval 40 ms + generation 17.0 s = 17.0 s   (qwen3:14b)



### 5.2 Ask your own question — edit this cell and re-run it

Put anything in `MY_QUESTION` and run the cell (`Ctrl`+`Enter`). Nothing above needs re-running;
the index stays in memory.

**Arabic works here too** — `ask()` detects the script and routes the question through §5.5's
translation path on its own, no flag to set.

`show_context=True` prints the exact chunks the model was handed, which is how you tell a
retrieval failure from a generation failure when an answer looks wrong. On an Arabic question it
also prints the English the query was translated into and the English answer the validator saw.

The corpus answers form numbers, step responsibilities, deadlines in days and hours,
definitions, and who approves what — across customer satisfaction, site branding, subcontract
agreements, quality inspection and testing, self-execution, project initiation, project
launching, vendor selection and procurement.

**Broad "explain the whole process" questions work, and they are slow.** Answers are no longer
length-capped (§5), so *"Explain to me the process operations in procurement"* returns all 18
steps — about 1,240 tokens, roughly 8 minutes on this CPU. That is the honest cost of not being
cut off mid-sentence. Ask a narrow question if you want a fast answer; the short lookups above
are 20–30 s once the model is warm.

In [8]:
MY_QUESTION = "What is the form number for risk assessment?"

_ = ask(MY_QUESTION, show_context=False)

Q: What is the form number for risk assessment?
  The form number for risk assessment is F-P-QD-05-01. This information is
  specified in the documents PTN02_Project_Launching_Process.pdf,
  PTN01_Project_Initiation_Process.pdf, and
  PVMO01_Vendor_selection_and_Bidding_Process.pdf.

  sources (top-3, route=none):
    [0.984] PTN02_Project_Launching_Process.pdf  |  PROCESS RISK ASSESSMENT
    [0.981] PTN01_Project_Initiation_Process.pdf  |  PROCESS RISK ASSESSMENT
    [0.966] PVMO01_Vendor_selection_and_Bidding_Process.pdf  |  PROCESS RISK ASSESSMENT

  cited forms : ['F-P-QD-05-01']   validator: clean
  latency     : retrieval 15 ms + generation 24.2 s = 24.2 s   (qwen3:14b)



### 5.3 Interactive prompt

A REPL, if you'd rather not edit a cell each time. Run the cell, type questions in either
language, and press Enter on a blank line (or type `quit`) to stop.

In [9]:
def chat():
    print(f"RME process chatbot | {MODEL} | ask in English or Arabic")
    print("blank line or 'quit' to exit\n")
    while True:
        try:
            q = input("you> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n(stopped)")
            return
        if not q or q.lower() in {"quit", "exit", "q"}:
            print("(stopped)")
            return
        try:
            ask(q)
        except Exception as e:
            # One bad question shouldn't drop you out of the session.
            print(f"  ! {type(e).__name__}: {e}\n")


try:
    chat()
except Exception as e:      # headless execution has no stdin
    print(f"interactive prompt unavailable here ({type(e).__name__}: {e}). "
          "Run this cell yourself in Jupyter or VS Code.")

RME process chatbot | qwen3:14b | ask in English or Arabic
blank line or 'quit' to exit

interactive prompt unavailable here (StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.). Run this cell yourself in Jupyter or VS Code.


### 5.4 The case that matters: a question the documents do not answer

The 9 documents say nothing about milestone penalties. A model answering from its own weights
would happily invent a clause and a form number to go with it. Retrieval still returns its three
nearest chunks — it always returns something — so the refusal has to come from the prompt, and
the validator independently confirms nothing was fabricated on the way out.

In [10]:
_ = ask("What is the penalty amount if a subcontractor misses a milestone date?")

Q: What is the penalty amount if a subcontractor misses a milestone date?
  Not specified in these process documents.

  sources (top-3, route=['PCN']):
    [0.730] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS RISK ASSESSMENT
    [0.684] PCN01_Subcontract_Agreement_Process_1.pdf  |  DOCUMENT CHANGE HISTORY
    [0.600] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS OPERATION

  cited forms : none   validator: clean
  latency     : retrieval 22 ms + generation 165.8 s = 165.8 s   (qwen3:14b)



### 5.5 Arabic questions

RME's users write in Arabic. The corpus does not — and neither do the chunk embeddings, the
BM25 tokeniser (`[a-z0-9]+`), the routing table in `chunker.PREFIX_HINTS`, or the `F-P-…` regex
in `validator.py`. Making all of that multilingual is a rebuild of the entire retrieval stack,
for nine documents.

So `translate.py` translates at the edges and leaves the middle alone:

```
  ما هو النموذج المستخدم لاستبيان رضا العملاء؟
      |  is_arabic()   fraction of *letters* written in Arabic script - the denominator
      |                is letters only, so a Latin form number inside the question
      v                cannot drag it under the threshold
  "What form is used for the customer satisfaction survey?"
      |
      |  the existing pipeline, untouched: route -> retrieve -> generate -> validate
      v
  "The Customer Satisfaction Survey uses form F-P-CM-01-01 ..."
      |  translate EN->AR, on the already-validated answer
      v
  "يستخدم استبيان رضا العملاء النموذج F-P-CM-01-01 ..."
```

`qwen3:14b` does the translating as well as the answering, so this adds no dependency, needs no
extra download, and nothing leaves the machine.

**Order matters in two places, and both are load-bearing.** Translation happens *before*
embedding, so retrieval, routing and BM25 all still see English and the measured 100% top-3
continues to apply. Validation happens *before* the answer is translated back, because the
whitelist check has to run against what the model actually emitted. The translation prompts also
pin form numbers, doc codes and filenames as verbatim Latin script — a form number re-rendered
in Arabic-Indic digits is a citation nobody can look up.

**Two costs, stated plainly.** An Arabic question makes three generation calls instead of one,
so expect roughly 2–3× the latency on a short answer; the breakdown below splits translate-in
and translate-out out from generation so you can see it rather than infer it. On a long
explanatory answer the multiplier is worse than 3×, because Arabic is token-denser than English
— the translate-out leg generates more tokens than the English answer it is translating. And the pipeline is now only as good
as the translation: a mistranslated domain term becomes a retrieval miss with nothing in the
output to signal that anything went wrong. That is why the record keeps `question_en` and
`show_context=True` prints it.

**What is not established here:** Arabic answer quality is unmeasured. The 36-question eval set
is English-only, so there is no Arabic equivalent of the accuracy number this notebook leans on
everywhere else. The three questions below are a demonstration, not a benchmark — an Arabic
eval set is the obvious next piece of work, and until it exists the honest claim is that the
path runs, not that it is accurate.

In [11]:
# The same three shapes as §5.1 and §5.4, asked in Arabic.

# expect F-P-CM-01-01
_ = ask("ما هو النموذج المستخدم لاستبيان رضا العملاء؟")

# expect 24 hours
_ = ask("خلال كم ساعة يجب على مدير المشروع إرسال خطط الإجراءات التصحيحية "
        "بعد الإبلاغ عن فجوة في رضا العملاء؟")

# unanswerable - the refusal has to survive the round trip too, and the
# validator must still report no fabricated citation
_ = ask("ما هي قيمة الغرامة إذا تأخر مقاول الباطن عن تاريخ الإنجاز؟")

Q: ما هو النموذج المستخدم لاستبيان رضا العملاء؟
  النموذج المستخدم لاستبيان رضا العملاء هو **استبيان رضا العملاء، F-P-
  CM-01-01**.   (المصدر: PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [0.931] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.868] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS INPUT
    [0.818] PCM01_Customer_Satisfaction_Process_1.pdf  |  RELATED DOCUMENTED INFORMATION

  cited forms : ['F-P-CM-01-01']   validator: clean
  latency     : translate-in 10.0 s + retrieval 13 ms + generation 18.6 s + translate-out 17.6 s = 46.2 s   (qwen3:14b)



Q: خلال كم ساعة يجب على مدير المشروع إرسال خطط الإجراءات التصحيحية بعد الإبلاغ عن فجوة في رضا العملاء؟
  يجب على مدير المشروع إرسال خطط الإجراء التصحيحية خلال 24 ساعة بعد الإبلاغ عن
  فجوة في رضا العملاء. (وثيقة: PCM01_Customer_Satisfaction_Process_1.pdf)

  sources (top-3, route=['PCM']):
    [1.000] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS OPERATION
    [0.661] PCM01_Customer_Satisfaction_Process_1.pdf  |  PROCESS CONTROL
    [0.583] PCM01_Customer_Satisfaction_Process_1.pdf  |  PERFORMANCE MEASURES

  cited forms : none   validator: clean
  latency     : translate-in 8.5 s + retrieval 13 ms + generation 19.4 s + translate-out 14.4 s = 42.3 s   (qwen3:14b)



Q: ما هي قيمة الغرامة إذا تأخر مقاول الباطن عن تاريخ الإنجاز؟
  لم يُحدد في وثائق هذه العملية.

  sources (top-3, route=['PCN']):
    [0.941] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS RISK ASSESSMENT
    [0.685] PCN01_Subcontract_Agreement_Process_1.pdf  |  PROCESS OPERATION
    [0.642] PCN01_Subcontract_Agreement_Process_1.pdf  |  DOCUMENT CHANGE HISTORY

  cited forms : none   validator: clean
  latency     : translate-in 7.0 s + retrieval 17 ms + generation 166.6 s + translate-out 5.9 s = 179.5 s   (qwen3:14b)



---

### What this demo establishes, and what it does not

**Shown:** the pipeline runs end to end on the real PDFs, on this hardware, against a local
`qwen3:14b` — no cloud call, no placeholder numbers. Re-chunking the PDFs reproduces the shipped
`chunks.json` exactly, the validator runs on every answer, latency is measured rather than
estimated, and Arabic questions traverse the same path with the translation hops timed
separately.

**Why `qwen3:14b`:** the 36-question × per-type eval recorded in `model_eval_results.json` —
100% overall, 100% on the `unanswerable` category, zero flagged citations, against a decision
rule fixed before the numbers came in. Read that with one caveat: `gemma4:e4b` scored
*identically* on those 36 questions and was faster on mean latency, so the eval set does not
actually separate the two — it is saturated. The choice of qwen3 rests on the larger model
being the safer default at 500–600 documents, not on a measured win, and a harder eval set
could reopen it. **The harness that produced those numbers is no longer in this repo**, so the
JSON is a record to read, not something this notebook can regenerate.

**Chunking is the part that changed most recently.** `adaptive_chunker.py` replaced literal
`SECTION_HEADERS` matching (§2). On these 9 documents it is a wash by design — retrieval stays
at 100% top-3 and the validator's form whitelist is byte-identical at 67 — so the argument for
it is not accuracy here, it is that the old approach could not survive a document written to a
different template. Two caveats measured while building it are worth carrying: the font-size
signal is nearly inert on this corpus (`numbering` fires 103 times against `font_size`'s 3, so
the layout half is largely untested here), and every threshold in it — 0.85 fuzzy, 0.45 cosine,
1.15 size ratio — is calibrated on these 9 documents.

**Not shown:** whether the Arabic path is any good (§5.5) — it runs, and that is all that has
been established. Also not shown: any behaviour at corpus scale. 9 documents of 500–600.

**Still missing, and it is the next thing to build:** step-level splitting. Both chunkers leave
`step` as `None` on every chunk — the old one had a stage meant to fill it that never matched
anything, the new one does not implement it at all. Harmless at 9 documents where sections are
small; at 500–600 it is what keeps a process step attached to its own form number.

**Scale caveats, unchanged from the handoff:** `PREFIX_HINTS` in `retriever.py` is a hand-written
keyword table — fine for 6 families and 9 documents, but at 500–600 it should be derived from
document titles or routing becomes the thing that silently loses recall. FAISS `IndexFlatIP`
stays exact and sub-millisecond to roughly 100k chunks. And the validator's whitelist only
becomes safe as a *blocking* control once the full corpus is indexed.